# Sudan Climate EDA
NASA POWER MERRA-2 Daily Data — Khartoum (2015–2026)

## 1. Data Loading & Date Parsing

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/sudan.csv')
df['Country'] = 'Sudan'
df['Date'] = pd.to_datetime(df['YEAR'] * 1000 + df['DOY'], format='%Y%j')
df['Month'] = df['Date'].dt.month
print(df.shape)
df.head()

## 2. Summary Statistics & Missing Value Report

In [ ]:
df.replace(-999, np.nan, inplace=True)

In [ ]:
print('Duplicate rows:', df.duplicated().sum())
df.drop_duplicates(inplace=True)

**Duplicates:** Any duplicate rows found are dropped. Duplicates in NASA POWER data are rare but can occur due to extraction errors.

In [ ]:
df.describe()

**Interpretation of Summary Statistics:**
- T2M is the highest among all five countries, averaging ~30°C, reflecting Khartoum's hot desert climate.
- T2M_RANGE is large, indicating extreme diurnal temperature swings typical of arid environments.
- PRECTOTCORR is very low on average with most days recording 0mm, consistent with Sudan's desert climate.
- RH2M is the lowest among all countries, confirming extremely dry conditions.

In [ ]:
missing = df.isna().sum()
missing_pct = (missing / len(df)) * 100
missing_report = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct.round(2)})
print(missing_report[missing_report['Missing %'] > 0])

**Missing Values:**
Columns with >5% missing values are flagged. In arid regions like Sudan, some parameters may have more missing values due to limited ground-truth data for satellite model calibration.

## 3. Outlier Detection & Basic Cleaning

In [ ]:
climate_cols = ['T2M', 'T2M_MAX', 'T2M_MIN', 'PRECTOTCORR', 'RH2M', 'WS2M', 'WS2M_MAX']

z_scores = df[climate_cols].apply(lambda col: np.abs(stats.zscore(col.dropna())))
outlier_counts = (z_scores > 3).sum()
print('Outlier counts per column (|Z| > 3):')
print(outlier_counts)

**Outlier Decision:**
Outliers are retained. Sudan's extreme heat events and rare but intense rainfall episodes are climatically significant and must be preserved for the extreme event analysis in Task 3.

In [ ]:
df[climate_cols] = df[climate_cols].fillna(method='ffill')
threshold = len(df.columns) * 0.7
df.dropna(thresh=int(threshold), inplace=True)
print('Shape after cleaning:', df.shape)

In [ ]:
import os
os.makedirs('../data', exist_ok=True)
df.to_csv('../data/sudan_clean.csv', index=False)
print('Exported to data/sudan_clean.csv')

## 4. Time Series Analysis

In [ ]:
monthly_temp = df.groupby('Month')['T2M'].mean()

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(monthly_temp.index, monthly_temp.values, marker='o', color='tomato')

warmest = monthly_temp.idxmax()
coolest = monthly_temp.idxmin()
ax.annotate(f'Warmest\n{monthly_temp[warmest]:.1f}°C', xy=(warmest, monthly_temp[warmest]),
            xytext=(warmest + 0.3, monthly_temp[warmest] + 0.3), fontsize=9, color='red')
ax.annotate(f'Coolest\n{monthly_temp[coolest]:.1f}°C', xy=(coolest, monthly_temp[coolest]),
            xytext=(coolest + 0.3, monthly_temp[coolest] - 0.5), fontsize=9, color='blue')

ax.set_title('Monthly Average Temperature — Sudan (2015–2026)')
ax.set_xlabel('Month')
ax.set_ylabel('Avg T2M (°C)')
ax.set_xticks(range(1, 13))
ax.set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'])
plt.tight_layout()
plt.show()

In [ ]:
monthly_rain = df.groupby('Month')['PRECTOTCORR'].sum() / df['YEAR'].nunique()

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(monthly_rain.index, monthly_rain.values, color='steelblue')

peak = monthly_rain.idxmax()
ax.annotate(f'Peak\n{monthly_rain[peak]:.1f}mm', xy=(peak, monthly_rain[peak]),
            xytext=(peak + 0.3, monthly_rain[peak] + 0.5), fontsize=9, color='navy')

ax.set_title('Monthly Average Total Precipitation — Sudan (2015–2026)')
ax.set_xlabel('Month')
ax.set_ylabel('Avg Total PRECTOTCORR (mm)')
ax.set_xticks(range(1, 13))
ax.set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'])
plt.tight_layout()
plt.show()

**Time Series Observations:**
- Sudan shows the most extreme temperature seasonality, peaking around May–June before the brief rainy season cools things slightly.
- Rainfall is concentrated in July–August only, with near-zero precipitation for the rest of the year.
- This pattern reflects a Sahel/desert climate with a very short and unreliable wet season — a major climate vulnerability.

## 5. Correlation & Relationship Analysis

In [ ]:
numeric_cols = ['T2M', 'T2M_MAX', 'T2M_MIN', 'T2M_RANGE', 'PRECTOTCORR', 'RH2M', 'WS2M', 'WS2M_MAX', 'PS', 'QV2M']
corr = df[numeric_cols].corr()

plt.figure(figsize=(12, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation Heatmap — Sudan')
plt.tight_layout()
plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.scatter(df['T2M'], df['RH2M'], alpha=0.3, color='teal', s=5)
ax1.set_title('T2M vs RH2M')
ax1.set_xlabel('T2M (°C)')
ax1.set_ylabel('RH2M (%)')

ax2.scatter(df['T2M_RANGE'], df['WS2M'], alpha=0.3, color='coral', s=5)
ax2.set_title('T2M_RANGE vs WS2M')
ax2.set_xlabel('T2M_RANGE (°C)')
ax2.set_ylabel('WS2M (m/s)')

plt.tight_layout()
plt.show()

**Three Strongest Correlations:**
1. **T2M & T2M_MAX / T2M_MIN** — Very strong positive correlation as expected.
2. **T2M & QV2M** — Strong positive correlation; desert air holds little moisture but spikes during the brief rainy season.
3. **T2M & RH2M** — Strong negative correlation in arid climates; as temperature rises, relative humidity drops sharply.

## 6. Distribution Analysis

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
rain_nonzero = df['PRECTOTCORR'][df['PRECTOTCORR'] > 0]
ax.hist(np.log1p(rain_nonzero), bins=50, color='steelblue', edgecolor='white')
ax.set_title('Distribution of PRECTOTCORR (log scale) — Sudan')
ax.set_xlabel('log(1 + PRECTOTCORR)')
ax.set_ylabel('Frequency')
plt.tight_layout()
plt.show()

**Precipitation Distribution:**
Sudan has the most extreme skew — the vast majority of days record 0mm. Even after log transformation, the distribution remains skewed, reflecting the desert climate where rainfall is rare and highly concentrated in a few intense events.

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(df['T2M'], df['RH2M'], s=df['PRECTOTCORR'] * 3 + 1,
            alpha=0.3, color='mediumseagreen', edgecolors='none')
plt.title('Bubble Chart: T2M vs RH2M (bubble size = PRECTOTCORR) — Sudan')
plt.xlabel('T2M (°C)')
plt.ylabel('RH2M (%)')
plt.tight_layout()
plt.show()